# Compare R137 / W138 dihedral state (v2)

v2 goals:
- No GUI (no tkinter)
- Load trajectories from the index CSV
- Cache per-simulation CSVs for expensive dihedral calculations
- Reproduce time-trace + histogram plots and the binarized timeline summary


In [ ]:
# Parameters

import os
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
if (_cwd / 'src').is_dir():
    REPO_ROOT = _cwd
elif (_cwd.parent / 'src').is_dir():
    REPO_ROOT = _cwd.parent
else:
    REPO_ROOT = Path(os.getenv('MD_REPO_ROOT', _cwd)).resolve()

SRC_DIR = REPO_ROOT / 'src'
if SRC_DIR.is_dir() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Index CSV
if 'MD_INDEX_CSV' in os.environ:
    INDEX_CSV = Path(os.environ['MD_INDEX_CSV']).expanduser().resolve()
else:
    _candidates = [
        Path(r'D:/Xiao Lab Dropbox/Lab Members/Yehya_Nico/Projects/MDfolder/FtsW Manuscript/Anton_SimulationIndex20260413.csv'),
        REPO_ROOT / 'data' / 'trajectory_index.csv',
    ]
    INDEX_CSV = next((p.resolve() for p in _candidates if p.exists()), _candidates[-1].resolve())

OUT_ROOT = Path(os.getenv('MD_OUT_ROOT', INDEX_CSV.parent)).expanduser().resolve()

# Choose residue + angle
RESI = int(os.getenv('MD_RESI', '137'))
ANGLE_TYPE = os.getenv('MD_ANGLE_TYPE', 'Psi')  # Psi / Phi / Chi1
VAR_NAME = f'R{RESI} {ANGLE_TYPE} Angle'
SHORT_VAR_NAME = f'R{RESI}_{ANGLE_TYPE}_Angle'

# Curated simulation preset + optional per-variable override file
PER_VAR_OVERRIDE_DIR = REPO_ROOT / 'notebooks' / 'curated'

# Plot/timeline parameters
X_LIM = (-420, 420)
MAX_HIST = 0.4
SCALE = 2.6
ANGLE_RANGE_FOR_TIMELINE = (-120, 120)

FORCE_RECOMPUTE = False
BUILD_17FULL = True

print(f'REPO_ROOT: {REPO_ROOT}')
print(f'INDEX_CSV: {INDEX_CSV}')
print(f'OUT_ROOT:  {OUT_ROOT}')
print(f'VAR_NAME:  {VAR_NAME}')


In [ ]:
# Imports

from curated_lists import curated_list_with_optional_override, persist_per_variable_override_if_changed
from traj_utils import validate_traj_index, read_trajectories
from dihedral_analysis import (
    compute_single_residue_dihedral,
    write_dihedral_csv,
    build_17full_dihedral_csv,
    DihedralPlotStyle,
    plot_dihedral_trace_hist,
    AngleRangeSummaryStyle,
    plot_angle_range_summary,
)

from pathlib import Path


In [ ]:
# Curated simulation list

default_sims = [
    '8','14','14b','15','15b','17','17ext','19','42','43','43b','43c',
    'A','B','C','D','E','F','38',
    '53','58',
]

override_path = PER_VAR_OVERRIDE_DIR / f'paper_main__{SHORT_VAR_NAME}.txt'
plot_list = curated_list_with_optional_override(default_sims, override_path)
persist_per_variable_override_if_changed(
    default_sims=default_sims,
    used_sims=plot_list,
    curated_dir=PER_VAR_OVERRIDE_DIR,
    short_var_name=SHORT_VAR_NAME,
)

print(f'plot_list ({len(plot_list)}): {plot_list}')


In [ ]:
# Load trajectories

valid_df = validate_traj_index(str(INDEX_CSV))

# Only load sims we need (plot_list + the segments needed for 17full)
sims_to_load = list(dict.fromkeys(plot_list + (['17','17ext'] if BUILD_17FULL else [])))
u_list, label_list, tf_list = read_trajectories(valid_df, sims_to_load)

# helper to map sim -> universe/time_factor in our loaded subset
sim_to_idx = {s: i for i, s in enumerate(sims_to_load)}
print(f'Loaded {len(u_list)} sims')


In [ ]:
# Compute/cache CSVs

out_dir = (OUT_ROOT / SHORT_VAR_NAME).resolve()
out_dir.mkdir(parents=True, exist_ok=True)

# FtsW segid overrides from the v1 notebook
FTSW_SEL_BY_SIM = {
    '42': 'segid PAG1',
    '43': 'segid PAU1',
    '43b': 'segid PAU1',
    '43c': 'segid PAU1',
}

data_dict = {}
for sim_number in plot_list:
    out_csv_path = out_dir / f'{sim_number}_{SHORT_VAR_NAME}.csv'
    if out_csv_path.exists() and not FORCE_RECOMPUTE:
        data_dict[sim_number] = str(out_csv_path)
        continue

    if sim_number not in sim_to_idx:
        print(f'Skipping {sim_number}: not loaded')
        continue

    i = sim_to_idx[sim_number]
    u = u_list[i]
    tf = float(tf_list[i])

    ftsw_sel = FTSW_SEL_BY_SIM.get(sim_number, 'segid PROD')

    try:
        wrapped, unwrapped = compute_single_residue_dihedral(
            u, resi=RESI, angle_type=ANGLE_TYPE, ftsw_sel=ftsw_sel
        )
        write_dihedral_csv(
            time_factor=tf,
            out_path=out_csv_path,
            var_name=VAR_NAME,
            wrapped=wrapped,
            unwrapped=unwrapped,
        )
        data_dict[sim_number] = str(out_csv_path)
    except Exception as e:
        print(f'Skipping {sim_number}: {type(e).__name__}: {e}')

print(f'Cached CSVs: {len(data_dict)}')


In [ ]:
# Build 17full (optional)

if BUILD_17FULL and ('17' in data_dict) and ('17ext' in data_dict):
    out_path = out_dir / f'17full_{SHORT_VAR_NAME}.csv'
    if (not out_path.exists()) or FORCE_RECOMPUTE:
        build_17full_dihedral_csv(
            csv_17=data_dict['17'],
            csv_17ext=data_dict['17ext'],
            out_path=out_path,
        )
    data_dict['17full'] = str(out_path)
    print(f'Built/loaded 17full: {out_path}')


In [ ]:
# Plot: time trace + histogram per simulation (writes PNG next to CSV)

style = DihedralPlotStyle(scale=SCALE, max_hist=MAX_HIST, x_lim=X_LIM)

for sim_number, csv_path in data_dict.items():
    png_path = Path(csv_path).with_suffix('.png')

    title = sim_number
    try:
        if sim_number in sim_to_idx:
            title = label_list[sim_to_idx[sim_number]]
    except Exception:
        pass

    plot_dihedral_trace_hist(
        csv_path=csv_path,
        var_name=VAR_NAME,
        out_png_path=png_path,
        title=title,
        style=style,
    )


In [ ]:
# Timeline summary: fraction of time in angle range

summary_list = ['8','14','14b','15','15b','17full','42','43','43b','43c']
summary_list = [s for s in summary_list if s in data_dict]

out_png = out_dir / f"{'-'.join(summary_list)}_{SHORT_VAR_NAME}_range_summary.png"

plot_angle_range_summary(
    data_dict=data_dict,
    out_png_path=str(out_png),
    summary_list=summary_list,
    sim_list=sims_to_load,
    label_list=label_list,
    var_name=VAR_NAME,
    angle_range=ANGLE_RANGE_FOR_TIMELINE,
    style=AngleRangeSummaryStyle(scale=4.0, long_end=True),
)
